# Convolution 2.0

## Import libraries

In [1]:
import shutil
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pypher
import warnings
from astropy.io import fits
from astropy.modeling import models, fitting
from astropy.nddata import block_reduce 
from astropy.utils.exceptions import AstropyWarning
from collections import defaultdict
from pathlib import Path

## Set directories

Determine file paths and obtain lists of galaxy images and PSF files for each survey inside Input/

In [2]:
CWD = Path.cwd()
ROOT = CWD.parents[1]
ASTROVELLO_DIR = ROOT / "AsTrovello_2_0"

input_dir = ASTROVELLO_DIR / "Input"

survey_paths = list(input_dir.glob("*"))
survey_names = [f.name for f in survey_paths]

galaxy = "ngc1097"

survey_names

['PHANGS', 'S4G']

# Configurations Dictionary ()

In [25]:
SURVEY_CONFIG = {
                    "PHANGS": 
                    {
                        "TELESCOP": "HST",
                        "INSTRUME": "WFC3",
                        "pixel_scale_arcsec": 0.0395,
                        "binned_factor": 4,
                        "unit_type": "electrons/s", # usado em units.py
                        "force_tan_sip": False,
                        "image_suffix": f"*{galaxy.lower()}*_exp-drc-sci.fits",
                        "psf_suffix": "*PSFSTD*.fits"
                    },
                    "S4G":
                    {
                        "TELESCOP": "Spitzer",
                        "INSTRUME": "IRAC",
                        "pixel_scale_arcsec": 
                        {
                            1: 1.221, # Channel 1
                            2: 1.223 # Channel 2
                        },
                        "binned_factor": 5,
                        "unit_type": "mjy/sr", # usado em units.py
                        "force_tan_sip": True,
                        "image_suffix": f"{galaxy.upper()}.phot.*.fits",
                        "psf_suffix": "*_col129_row129.fits"
                    }
                }

# Drivers (Classes)

## Base Driver (Father Class)

In [108]:
# 1. A CLASSE PAI (Guarda o que é GENÉRICO para todos os surveys)
class BASE_Driver:
    def __init__(self, config_dict: dict):
        self.config = config_dict

    def get_files(self, dir_path: Path, mode: str) -> list:
        # A busca via glob agora mora apenas AQUI no pai!
        suffix_key = f"{mode}_suffix"
        
        if suffix_key in self.config:
            # O f-string no glob permite injetar variáveis caso o sufixo use o nome da galáxia!
            pattern = self.config[suffix_key]
            return list(dir_path.glob(pattern))
            
        raise ValueError(f"Mode '{mode}' not configured for {self.__class__.__name__}.")

    def get_survey(self, file_path: Path) -> str:
        AVAILABLE_SURVEYS = self.config.keys()
        for survey in AVAILABLE_SURVEYS:
            if survey in str(file_path):
                return survey

    def get_pixel_scale(self, filter_name: str) -> float:
        # Padrão genérico: se for um valor simples no dicionário, já resolve aqui no pai!
        return self.config["pixel_scale_arcsec"]

    @property
    def get_binned_factor(self) -> int:
        return self.config.get("binned_factor", 1)

    def get_psf_pixel_scale(self, filter_name: str) -> float:
    # Pega a escala nativa do survey/canal e divide pelo binned_factor
        raw_scale = self.get_pixel_scale(filter_name)
        return raw_scale / self.get_binned_factor

## Phangs Driver (Inherits from Base_Driver)

In [109]:
# 2. OS DRIVERS FILHOS (Herdam do pai e só escrevem o que for ESPECÍFICO)
class PHANGS_Driver(BASE_Driver):
    """Herda get_files e get_pixel_scale de BaseDriver."""
    
    def get_filter_name(self, filename: str) -> str:
        return filename.replace('.fits', '').split('_')[-1].lower()

## S4G Driver (Inherits from Base_Driver)

In [110]:
class S4G_Driver(BASE_Driver):
    """Herda get_files de BaseDriver, mas sobrescreve o que é peculiar do S4G."""
    
    def get_filter_name(self, filename: str) -> str:
        if 'IRAC1' in filename: return 'irac1'
        if 'IRAC2' in filename: return 'irac2'
        return 'unknown'

    def get_pixel_scale(self, filter_name: str) -> float:
        # Sobrescreve o método do pai apenas porque o S4G tem escalas diferentes por canal!
        channel = 1 if filter_name == 'irac1' else 2
        return self.config["pixel_scale_arcsec"][channel]

## Obtain all survey files (Science images and PSFs)

In [ ]:
# 1. Dicionário Mapeando os Drivers ativos
DRIVERS = {
    "BASE": BASE_Driver(config_dict = SURVEY_CONFIG),
    "PHANGS": PHANGS_Driver(config_dict = SURVEY_CONFIG["PHANGS"]),
    "S4G": S4G_Driver(config_dict = SURVEY_CONFIG["S4G"])
}

image_files = []
psf_files = []

# 2. Loop 100% Genérico (Sem NENHUM 'if/elif')
for survey in survey_names:
    if survey not in DRIVERS:
        print(f"Aviso: Survey '{survey}' não possui um driver configurado. Pulando...")
        continue

    # Puxa o driver correto para o survey atual
    driver = DRIVERS[survey]
    
    image_dir = input_dir / survey / "galaxies" / galaxy
    psf_dir = input_dir / survey / "PSF"

    # Não importa qual survey é, a chamada é rigorosamente a mesma!
    current_image_files = driver.get_files(dir_path = image_dir, mode = "image")
    current_psf_files = driver.get_files(dir_path = psf_dir, mode = "psf") # Nota: PSF costuma usar psf_dir!

    image_files.extend(current_image_files)
    psf_files.extend(current_psf_files)

In [112]:
image_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f555w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f814w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.1.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.2.fits')]

In [113]:
psf_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F275W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F336W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F438W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F555W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/PSF/PSFSTD_WFC3UV_F814W.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC1_col129_row129.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/PSF/IRAC2_col129_row129.fits')]

## Determine PSF resolutions 
Calculate FWHM to determine each surveys resolution and define the master file for convolution (lowest resolution).

In [114]:
phangs_img_header = fits.getheader(image_files[2], 0) 
phangs_img_header["INSTRUME"]

'IRAC'

In [115]:
supported_instruments = [survey_data["INSTRUME"] for survey_data in SURVEY_CONFIG.values()]
psf_files[0].name

'PSFSTD_WFC3UV_F275W.fits'

### Get FWHM function

In [116]:
def get_fwhm(data: np.ndarray) -> float:
    """Estimates the Full Width at Half Maximum (FWHM) using a 2D Gaussian fit.
    
    This method is robust against background noise, negative pixels, and large 
    image bounding boxes. It utilizes the Levenberg-Marquardt least squares 
    algorithm to fit a 2D Gaussian model to the data. To account for slightly 
    elliptical PSFs, the effective sigma is calculated as the geometric mean 
    of the X and Y standard deviations.

    Args:
        data (np.ndarray): A 2D array representing the Point Spread Function (PSF) image.

    Returns:
        float: The estimated FWHM measured in pixels.
    """
    # 1. Remove qualquer NaN que possa quebrar o algoritmo
    data_clean = np.nan_to_num(data, nan=0.0)
    
    # 2. Cria uma malha de coordenadas X e Y do mesmo tamanho da imagem
    y, x = np.mgrid[:data_clean.shape[0], :data_clean.shape[1]]
    
    # 3. Estima os parâmetros iniciais (chutes) para ajudar o algoritmo a convergir mais rápido
    max_val = np.max(data_clean)
    y_center, x_center = np.unravel_index(np.argmax(data_clean), data_clean.shape)
    
    # Cria o modelo inicial da Gaussiana
    g_init = models.Gaussian2D(amplitude=max_val, x_mean=x_center, y_mean=y_center, 
                               x_stddev=2.0, y_stddev=2.0)
    
    # 4. Inicializa o algoritmo de ajuste (Levenberg-Marquardt Mínimos Quadrados)
    fit_g = fitting.LevMarLSQFitter()
    
    # 5. Ajusta o modelo aos dados
    with warnings.catch_warnings():
        # Ignora avisos inofensivos do astropy caso a PSF seja muito ruidosa
        warnings.simplefilter('ignore')
        g_fit = fit_g(g_init, x, y, data_clean)
    
    # 6. Extrai o desvio padrão (sigma) do eixo X e Y e tira a média geométrica
    # A média geométrica lida melhor com PSFs ligeiramente elípticas
    sigma_eff = np.sqrt(abs(g_fit.x_stddev.value * g_fit.y_stddev.value))
    
    # 7. Converte Sigma para FWHM em pixels
    fwhm_pixels = 2.3548 * sigma_eff
    
    return float(fwhm_pixels)

### Calculate FWHM function (for file list)

In [117]:
def calculateFWHM(psf_file_list: list, drivers: dict) -> tuple[dict, list]:
    """AsTrovello 2.0
    
    Iterates through a folder of PSF files, filters them by survey,
    and returns dictionaries containing their physical FWHM (in arcsec).

    Determines to which survey the file belongs based on its parent directory
    and extracts the filter name. Then, determines the PSF's binning factor 
    and its correct pixel scale from the SURVEY_CONFIG dictionary. Finally, 
    calculates the FWHM for each filter and returns a FWHM dictionary and a 
    list with valid file names. 

    Args:
        psf_file_list (list): List of PSF files paths.
        SURVEY_CONFIG (dict): Survey configurations dictionary.

    Returns:
        tuple: A tuple containing:
            - FWHM_dict (dict): Dictionary with filter names as keys and FWHM (in arcsec) as values.
            - valid_files (list): List of valid file names (where FWHM was successfully calculated).

    Note:
        For 3D PSF files, the FWHM is calculated over the mean PSF of the cube.
    """
    FWHM_dict, valid_files = {}, []
    
    # Silencia os avisos chatos de cabeçalho do Astropy
    warnings.simplefilter('ignore', category=AstropyWarning)
    
    for file in psf_file_list:
        if file.name.startswith('.'):
            continue
        
        str_file = str(file)
        survey = drivers["BASE"].get_survey(file_path = str_file)
        if not survey or survey not in drivers:
            continue

        driver = drivers[survey]

        filter_name = driver.get_filter_name(filename = str_file)
        psf_pixscale = driver.get_psf_pixel_scale(filter_name = filter_name)

        try:
            with fits.open(file, ignore_missing_end=True, ignore_missing_simple=True) as hdu:
                data = next((h.data for h in hdu if h.data is not None), None)
                
                if data is not None:
                    if data.ndim == 3: 
                        data = np.mean(data, axis=0) 
                    
                    # Usa o ajuste Gaussiano (ou a função robusta) que retorna em pixels
                    fwhm_pixels = get_fwhm(data)
                    
                    # Converte para escala física (arcsec) usando a escala de pixel superamostrada
                    FWHM_dict[filter_name] = np.float32(fwhm_pixels * psf_pixscale)
                    
                    valid_files.append(file.name)
                    print(f"Successfully read: {filter_name} (FWHM: {FWHM_dict[filter_name]:.4f} arcsec)")
                    
        except Exception as e:
            print(f"Processing error {file.name}: {e}")
            
    # Restaura os avisos para o resto do seu código
    warnings.simplefilter('default', category=AstropyWarning)
    
    return FWHM_dict, valid_files

In [118]:
DRIVERS["BASE"].get_survey(file_path = psf_files[0])

'PHANGS'

### 3. Determine lowest resolution

In [119]:
fwhm_dict, valid_files = calculateFWHM(psf_files, DRIVERS)
df_fwhm = pd.DataFrame(list(fwhm_dict.items()), columns=["Filter", "FWHM_arcsec"])
df_fwhm = df_fwhm.sort_values(by="FWHM_arcsec").reset_index(drop=True)

print("\nResolutions Table:\n", df_fwhm)

psf_master_name = df_fwhm.iloc[-1]['Filter']
print(f"\n==> Recommended PSF (master): {psf_master_name}")

Successfully read: f275w (FWHM: 0.0766 arcsec)
Successfully read: f336w (FWHM: 0.0810 arcsec)
Successfully read: f438w (FWHM: 0.0838 arcsec)
Successfully read: f555w (FWHM: 0.0825 arcsec)
Successfully read: f814w (FWHM: 0.0790 arcsec)
Successfully read: irac1 (FWHM: 1.5620 arcsec)
Successfully read: irac2 (FWHM: 1.5253 arcsec)

Resolutions Table:
   Filter  FWHM_arcsec
0  f275w     0.076603
1  f814w     0.079008
2  f336w     0.081011
3  f555w     0.082506
4  f438w     0.083771
5  irac2     1.525301
6  irac1     1.561972

==> Recommended PSF (master): irac1


## Clean PSFs

In [ ]:
def final_clean_psf(input_file, output_file, SURVEY_CONFIG):
    """
    Standardizes PSF headers and performs true downsampling for PyPHER compatibility.
    Calculates pixel scales, bins down oversampled PSFs, and ensures correct 
    centering and coordinate keywords.
    """
    if 'WFC3UV' in input_file:
        # HST scale: native 0.0395"/pix. We will bin down the 4x oversampled data.
        pixel_scale_arcsec = SURVEY_CONFIG["PHANGS"]["pixel_scale_arcsec"]
        pixel_scale_deg = pixel_scale_arcsec / 3600.0

        with fits.open(input_file, ignore_missing_end=True) as hdu:
            # Average the PSF cube to get a 2D representative PSF
            data_2d = np.mean(hdu[0].data, axis=0)
            
            # Downsample the array by a factor of 4 (summing blocks of 4x4 pixels)
            data_2d_binned = block_reduce(data_2d, block_size=4, func=np.sum)
            
            # Force odd parity: PyPHER prefers kernels/PSFs with an odd number of pixels
            if data_2d_binned.shape[0] % 2 == 0:
                data_2d_binned = data_2d_binned[:-1, :-1]
                
            # Normalize to ensure flux conservation
            data_2d_binned = data_2d_binned / np.sum(data_2d_binned)

            new_hdu = fits.PrimaryHDU(data_2d_binned)
            
            # Inject WCS keywords required by PyPHER/Astropy
            new_hdu.header.update({
                'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
                'CRVAL1': 0.0, 'CRVAL2': 0.0,
                'CRPIX1': (data_2d_binned.shape[1] // 2) + 1, 'CRPIX2': (data_2d_binned.shape[0] // 2) + 1,
                'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
                'PIXSCALE': pixel_scale_arcsec
            })
            new_hdu.writeto(output_file, overwrite=True)
            print(f"==> File ready to be applied in PyPHER (Binned 4x): {os.path.basename(output_file)}")

    elif any(x in input_file for x in ['IRAC1', 'IRAC2']):
        # Spitzer scale: native ~1.22"/pix. We will bin down the 5x oversampled data.
        pixel_scale_arcsec = 1.221 if 'IRAC1' in input_file else 1.213
        pixel_scale_deg = pixel_scale_arcsec / 3600.0

        with fits.open(input_file) as hdu:
            data_raw = hdu[0].data
            
            # Downsample the array by a factor of 5
            data_2d_binned = block_reduce(data_raw, block_size=5, func=np.sum)
            
            # Force odd parity
            if data_2d_binned.shape[0] % 2 == 0:
                data_2d_binned = data_2d_binned[:-1, :-1]
                
            # Normalize to ensure flux conservation
            data_2d_binned = data_2d_binned / np.sum(data_2d_binned)

            new_hdu = fits.PrimaryHDU(data_2d_binned)
            new_hdu.header.update({
                'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
                'CRVAL1': 0.0, 'CRVAL2': 0.0,
                'CRPIX1': (data_2d_binned.shape[1] // 2) + 1, 'CRPIX2': (data_2d_binned.shape[0] // 2) + 1,
                'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
                'PIXSCALE': pixel_scale_arcsec
            })
            new_hdu.writeto(output_file, overwrite=True)
            print(f"==> File ready to be applied in PyPHER (Binned 5x): {os.path.basename(output_file)}")


### 1. Identify survey

In [38]:
def identify_psf_survey(filename: Path, SURVEY_CONFIG: dict) -> tuple[float, int]:
    full_filename = str(filename).lower()
    short_filename = filename.name.lower()

    if ("phangs" in full_filename) or ("wfc3" in full_filename):
        survey = "PHANGS"
        pixel_scale_arcsec = SURVEY_CONFIG[survey]["pixel_scale_arcsec"]
        binned_factor = SURVEY_CONFIG[survey]["binned_factor"]
    elif ("s4g" in full_filename) or ("irac" in full_filename):
        survey = "S4G"
        channel = short_filename.split("_")[0].replace("irac", "")
        channel = int(channel)
        pixel_scale_arcsec = SURVEY_CONFIG[survey]["pixel_scale_arcsec"][channel]
        binned_factor = SURVEY_CONFIG[survey]["binned_factor"]
    else:
        raise ValueError(f"Survey unidentified! Please provide available survey for: {filename}")

    return float(pixel_scale_arcsec), int(binned_factor)

### 2. Clean PSF using respective survey info

In [ ]:
def clean_psf(input_file: str, output_file: str, pixel_scale_arcsec: float, binned_factor: int):
    """Standardizes PSF headers and performs true downsampling for PyPHER compatibility.
    
    Agnostic function that calculates pixel scales, bins down oversampled PSFs, 
    forces odd parity, normalizes flux, and ensures correct centering and WCS keywords.

    Args:
        input_file (str): Path to the input PSF FITS file.
        output_file (str): Path to save the cleaned PSF.
        pixel_scale_arcsec (float): Native pixel scale of the instrument.
        binned_factor (int): Factor by which the PSF is oversampled.
        is_3d (bool): If True, averages the data along axis 0 to create a 2D representation.
    """
    pixel_scale_deg = pixel_scale_arcsec / 3600.0

    with fits.open(input_file, ignore_missing_end=True, ignore_missing_simple=True) as hdu:
        data = next((h.data for h in hdu if h.data is not None), None)
        
        if data is None:
            print(f"==> Error: No valid data found in {input_file}")
            return

        # 1. Trata cubos 3D (ex: WFC3 do Hubble)
        if data.ndim == 3:
            data = np.mean(data, axis=0)

        # 2. Aplica o downsampling apenas se o fator for maior que 1
        if binned_factor > 1:
            data_processed = block_reduce(data, block_size=binned_factor, func=np.sum)
        else:
            data_processed = data.copy()

        # 3. Força paridade ímpar para o PyPHER
        if data_processed.shape[0] % 2 == 0:
            data_processed = data_processed[:-1, :-1]
            
        # 4. Normaliza para assegurar conservação de fluxo
        data_processed = data_processed / np.sum(data_processed)

        # 5. Criação do novo FITS e injeção do WCS
        new_hdu = fits.PrimaryHDU(data_processed)
        new_hdu.header.update({
            'CTYPE1': 'RA---TAN', 'CTYPE2': 'DEC--TAN',
            'CRVAL1': 0.0, 'CRVAL2': 0.0,
            'CRPIX1': (data_processed.shape[1] // 2) + 1, 'CRPIX2': (data_processed.shape[0] // 2) + 1,
            'CDELT1': -pixel_scale_deg, 'CDELT2': pixel_scale_deg,
            'PIXSCALE': pixel_scale_arcsec
        })
        
        new_hdu.writeto(output_file, overwrite=True)
        print(f"==> File ready for PyPHER (Binned {binned_factor}x): {os.path.basename(output_file)}")

## Pypher kernel creation

In [ ]:
def pypher_kernel_creation(fwhm_dict: dict, psf_master_path: Path, input_dir: Path, output_dir: Path) -> str:
    """
    Prepares a list of shell commands for the PyPHER library to generate homogenization kernels.
    It identifies which PSF belongs to which survey to locate files in the 'PSF_CLEAN' folders.
    """
    psf_path_phangs_clean = input_dir / 'PHANGS' / 'PSF_CLEAN'
    psf_path_s4g_clean = input_dir / 'S4G' / 'PSF_CLEAN'
    psf_master_name = psf_master_path.stem.split('_')[0].lower()

    if os.path.exists(output_dir):
        print(f"==>  Removing previous directory: {os.path.basename(output_dir)}")
        shutil.rmtree(output_dir)
    os.makedirs(output_dir)

    comandos_pypher = []
    for filtro in fwhm_dict.keys():
        if filtro == psf_master_name: continue
        
        # Determine High-Resolution PSF path
        if filtro.startswith('f'):
            psf_high_res = os.path.join(psf_path_phangs_clean, f"PSFSTD_WFC3UV_{filtro.upper()}.fits")
        elif filtro.startswith('i'):    
            psf_high_res = os.path.join(psf_path_s4g_clean, f"{filtro.upper()}_col129_row129.fits")
        else: continue

        kernel_name = os.path.join(output_dir, f"kernel_{filtro}_to_{psf_master_name}.fits")
        # Format command: pypher [HR_PSF] [Target_PSF] [Output_Kernel]
        comandos_pypher.append(f"pypher {psf_high_res} {psf_master_path} {kernel_name}")

    return str(comandos_pypher)

## Create convolution dictionary

In [ ]:
def convolved_dict(path_phangs, path_s4g_reprojected, path_kernels, error = False):
    """
    Organizes all images and kernels into a nested dictionary indexed by filter.
    Used to pair the correct kernel with its corresponding image for convolution.
    """
    if error:
        phangs_files = list(path_phangs.glob('*err-drc-wht.fits'))
        s4g_files = list(path_s4g_reprojected.glob('*_error.fits'))
    else:
        phangs_files = list(path_phangs.glob('*exp-drc-sci.fits'))
        all_s4g_files = list(path_s4g_reprojected.glob('*.fits'))
        s4g_files = [f for f in all_s4g_files if '_error' not in f.name]
        
    kernel_files = list(path_kernels.glob('*.fits'))

    all_files = phangs_files + s4g_files + kernel_files
    filter_info = [f.name.split('_')[1] for f in sorted(kernel_files)]

    if error:
        fftconvolve_dict = defaultdict(lambda: {'kernel': {}, 'err_img': {}}) 
        for f in filter_info:
            for file in all_files:
                file_string = str(file).lower()
                if f in file_string:
                    key = 'kernel' if 'kernel' in file_string else 'err_img'
                    fftconvolve_dict[f][key]['path'] = file
                    fftconvolve_dict[f][key]['name'] = file.name

    else:
        fftconvolve_dict = defaultdict(lambda: {'kernel': {}, 'img': {}}) 
        for f in filter_info:
            for file in all_files:
                file_string = str(file).lower()
                if f in file_string:
                    key = 'kernel' if 'kernel' in file_string else 'img'
                    fftconvolve_dict[f][key]['path'] = file
                    fftconvolve_dict[f][key]['name'] = file.name
    return fftconvolve_dict